# CrewAI: Multi-Agent Research Crews
## A Hands-On Workshop

**CrewAI** is a framework for building teams of AI agents that collaborate to complete complex tasks.
Instead of wiring a state machine by hand (as you would in LangGraph), you describe *who* each agent is,
*what* they need to do, and let the framework coordinate execution.

By the end of this workshop you will understand the three core primitives — `Agent`, `Task`, and `Crew` —
and have built a two-agent research crew that searches the web and produces a polished report.

---

### Workshop Roadmap

| # | Topic |
|---|-------|
| 1 | **Concepts** — What is CrewAI and how does it relate to LangGraph? |
| 2 | **Agents** — Defining identity: role, goal, backstory |
| 3 | **Tools** — Giving agents abilities with `@tool` |
| 4 | **Tasks** — Defining work: description, expected output, context |
| 5 | **Crew** — Orchestrating agents through tasks |
| 6 | **Run the Crew** — Kickoff and inspect results |
| 7 | **Process Modes** — Sequential vs hierarchical execution |
| ★ | **Extensions** — Custom tools, memory, async (bonus) |

---

### Prerequisites
- Python 3.10+ (a `.venv` with the requirements already installed)
- An `OPENAI_API_KEY` set in `.env` (or Colab Secrets)

### Key References
> Wu, Q., Bansal, G., et al. (2023). *AutoGen: Enabling Next-Gen LLM Applications via Multi-Agent Conversation.* https://arxiv.org/abs/2308.08155  
> Hong, S., Zheng, X., et al. (2023). *MetaGPT: Meta Programming for A Multi-Agent Collaborative Framework.* https://arxiv.org/abs/2308.00352  
> CrewAI documentation: https://docs.crewai.com  
> CrewAI GitHub: https://github.com/crewAIInc/crewAI

In [ ]:
# Install dependencies — runs only on Google Colab.
# Local users: your .venv already has everything from requirements.txt.
import sys


def _in_colab():
    try:
        import google.colab

        return True
    except ImportError:
        return False


if _in_colab():
    import subprocess

    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "crewai",
            "ddgs",
            "python-dotenv",
        ]
    )
    print("Colab install complete.")
else:
    print("Local environment detected — skipping install.")

In [ ]:
import os

# ── API key ───────────────────────────────────────────────────────────────────
# Colab: set in Secrets panel (left sidebar key icon)
# Local: create a .env file containing  OPENAI_API_KEY=sk-...
try:
    from google.colab import userdata

    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from dotenv import load_dotenv

    load_dotenv()

# ── Sanity check ──────────────────────────────────────────────────────────────
key = os.environ.get("OPENAI_API_KEY", "")
assert key.startswith("sk-"), (
    "OPENAI_API_KEY missing or invalid. "
    "Local: add it to .env.  Colab: Secrets panel -> Add secret 'OPENAI_API_KEY'"
)
print(f"OPENAI_API_KEY present: True  ({key[:8]}...)")

---

## Part 1 — What Is CrewAI and Why Does It Exist?

---

### The Problem With Single Agents

A single LLM agent running in a loop works well for focused tasks but breaks down when a job
requires multiple specialisations — research, writing, editing, coding — or when tasks are too
long to fit in a single context window. Multi-agent systems solve this by **dividing work across
specialised agents** that each maintain a focused identity.

CrewAI formalises this pattern: you declare *who* each agent is (role + backstory), *what* they
must produce (tasks with expected outputs), and *how* they share information (context chains).

---

### CrewAI vs LangGraph — Two Philosophies

| | LangGraph | CrewAI |
|---|---|---|
| **Mental model** | State machine — wire nodes and edges | Role-based crew — describe who does what |
| **You define** | Graph structure, state schema, edge conditions | Agents, tasks, and their relationships |
| **Flow control** | Explicit — you own every edge | Implicit — CrewAI orchestrates |
| **Branching** | Full conditional branching, loops, HITL | Sequential or hierarchical via `process=` |
| **Best for** | Fine-grained control, complex DAGs | Role-based pipelines, parallel crews |
| **Learning curve** | Higher — must understand graph theory | Lower — reads like a job description |

Neither is better — they solve different problems. Use LangGraph when you need precise flow control.
Use CrewAI when you want to describe roles and let agents figure out the steps.

---

### The Three Core Primitives

| Primitive | What it represents | Key fields |
|-----------|-------------------|------------|
| `Agent` | *Who* is doing the work | `role`, `goal`, `backstory`, `tools` |
| `Task` | *What* needs to be produced | `description`, `expected_output`, `agent`, `context` |
| `Crew` | Orchestrates the whole job | `agents`, `tasks`, `process` |

The `context` parameter on a `Task` is the key to collaboration:
it passes one task's output as input to the next.

### Research Crew Architecture

```
CREW  (process=sequential)
──────────────────────────────────────────────────────────────

  START
    │
    ▼
  ┌───────────────────────────────────────────────────────┐
  │  Agent: Researcher                                    │
  │  goal:  find accurate, recent information             │
  │  tools: [web_search]                                  │
  └───────────────────────┬───────────────────────────────┘
                          │  Task: research_task
                          │  expected: numbered list of
                          │  5-7 findings with sources
                          │
                          ▼  context passed ──────────────┐
  ┌───────────────────────────────────────────────────────┤
  │  Agent: Writer                                        │
  │  goal:  turn findings into a structured report        │◄┘
  │  tools: (none — works from researcher's output)       │
  └───────────────────────┬───────────────────────────────┘
                          │  Task: write_task
                          │  expected: 200-word report
                          │  with intro / key points / conclusion
                          │
                          ▼
                        END  → crew.kickoff() returns final report
──────────────────────────────────────────────────────────────
```

> **Key insight:** The `context=[research_task]` parameter on `write_task` is what makes this a
> *crew* rather than two independent agents. The writer receives the researcher's full output
> before generating the report.

---

## Part 2 — Agents: Defining Who Does the Work

---

An `Agent` in CrewAI has three identity fields that shape its system prompt:

| Field | Purpose | Effect on the LLM |
|-------|---------|-------------------|
| `role` | Job title | Appears in the system prompt header |
| `goal` | What success looks like | Drives decision-making during task execution |
| `backstory` | Personality and expertise | Tunes tone, depth, and perspective |
| `tools` | Callable abilities | Determines what external actions are available |
| `verbose` | Debug logging | Prints the agent's reasoning chain step-by-step |
| `llm` | Override model | Default: `gpt-4o`. Pass `llm="gpt-4o-mini"` to cut cost |

The combination of `role + goal + backstory` is equivalent to a carefully written system prompt.
Write them as if briefing a new team member: concrete, specific, and action-oriented.

In [ ]:
# ─── 2-A: Define the Researcher agent ────────────────────────────────────────
# No tools yet — just identity. We attach tools in Part 3.

from crewai import Agent

researcher = Agent(
    role="Senior Research Analyst",
    goal="Find accurate, recent information and synthesise it into clear findings",
    backstory=(
        "You are a meticulous researcher with a talent for cutting through noise. "
        "You always cite your sources and flag uncertainty when evidence is thin."
    ),
    verbose=True,
)

print(f"Agent created: {researcher.role}")
print(f"Goal: {researcher.goal}")

In [ ]:
# ─── 2-B: Define the Writer agent ────────────────────────────────────────────
# The writer has no search tools — it only works from what the researcher provides.
# That specialisation is intentional: one agent searches, one agent writes.

writer = Agent(
    role="Technical Writer",
    goal="Turn research findings into a clear, structured report for a developer audience",
    backstory=(
        "You are a concise technical writer who excels at structure and clarity. "
        "You never invent facts — you only synthesise what the researcher provides."
    ),
    verbose=True,
)

print(f"Agent created: {writer.role}")
print(f"Goal: {writer.goal}")

In [ ]:
# ─── 2-C: Inspect agent configuration ────────────────────────────────────────
# Before attaching tools, confirm both agents are configured as expected.

for agent in [researcher, writer]:
    print(f"{'─' * 50}")
    print(f"Role:      {agent.role}")
    goal_display = agent.goal[:70] + "..." if len(agent.goal) > 70 else agent.goal
    print(f"Goal:      {goal_display}")
    tools_display = [t.name for t in agent.tools] if agent.tools else "(none yet)"
    print(f"Tools:     {tools_display}")
print("─" * 50)

---

## Part 3 — Tools: Giving Agents Abilities

---

An agent without tools can only reason over its training data.
Tools let agents interact with the world: search the web, read files, call APIs, run code.

CrewAI tools are defined with the `@tool` decorator:
- The **function name** becomes the tool name the LLM sees
- The **docstring** becomes the tool description — the LLM reads it to decide *when* to call the tool
- The **type-annotated parameters** define what the LLM must supply

### Built-in vs Custom Tools

| Tool source | When to use |
|-------------|-------------|
| `crewai_tools.SerperDevTool` | Production web search (requires Serper API key) |
| `crewai_tools.FileReadTool` | Read local files |
| `crewai_tools.CodeInterpreterTool` | Run Python code |
| **`@tool` custom** | Any Python function — APIs, databases, calculations |

This workshop uses a custom `web_search` tool backed by DuckDuckGo — no extra API key required.

In [ ]:
# ─── 3-A: Define a web search tool ───────────────────────────────────────────
# The docstring is critical — it tells the LLM exactly when to invoke this tool.

from crewai.tools import tool


@tool("Web Search")
def web_search(query: str) -> str:
    """Search the web for recent information about a topic.
    Use for current events, framework comparisons, and facts that may postdate training.
    Input: a search query string.
    Output: a newline-separated list of result snippets.
    """
    from ddgs import DDGS

    results = list(DDGS().text(query, max_results=5))
    return "\n".join(f"- {r['body']}" for r in results) if results else "No results found."


print(f"Tool name: {web_search.name}")
print(f"Tool description (first 120 chars): {web_search.description[:120]}...")

In [ ]:
# ─── 3-B: Test the tool before attaching it to an agent ──────────────────────
# Always validate tools independently — debugging a bad tool inside a Crew is painful.

test_result = web_search.run("CrewAI multi-agent framework 2025")
print("Tool test result (first 400 chars):")
print(test_result[:400])

In [ ]:
# ─── 3-C: Attach the tool to the researcher ──────────────────────────────────
# Rebuild the researcher with the web_search tool attached.
# The writer intentionally gets no tools — it only synthesises from context.

researcher = Agent(
    role="Senior Research Analyst",
    goal="Find accurate, recent information and synthesise it into clear findings",
    backstory=(
        "You are a meticulous researcher with a talent for cutting through noise. "
        "You always cite your sources and flag uncertainty when evidence is thin."
    ),
    tools=[web_search],
    verbose=True,
)

print(f"Researcher tools: {[t.name for t in researcher.tools]}")
writer_tools = [t.name for t in writer.tools] if writer.tools else "(none — by design)"
print(f"Writer tools:     {writer_tools}")

---

## Part 4 — Tasks: Defining What Needs to Be Produced

---

An `Agent` defines *who* someone is. A `Task` defines *what* they must deliver.

### Task Fields

| Field | Required | Purpose |
|-------|----------|---------|
| `description` | Yes | Instructions the agent follows — like a project brief |
| `expected_output` | Yes | Format and quality bar for the result |
| `agent` | Yes | Which agent is responsible |
| `context` | No | List of prior tasks whose output this task receives |
| `output_file` | No | Write result to disk automatically |
| `human_input` | No | Pause for human review before the next task |

### The `context` Parameter — How Agents Collaborate

```
research_task  ─── output: numbered findings list
                         │
                         │  context=[research_task]
                         ▼
write_task     receives the findings and writes the report
```

Without `context`, each task runs blind — the writer would have no facts to work with.
With `context`, CrewAI injects the prior task's output into the next task's prompt automatically.

In [ ]:
# ─── 4-A: Define the research task ───────────────────────────────────────────

from crewai import Task

TOPIC = "LangGraph vs CrewAI for production AI agents in 2025"

research_task = Task(
    description=(
        f"Research the topic: '{TOPIC}'. "
        "Use web search to gather 5-7 key facts. "
        "Include specific version numbers, benchmark results, or adoption metrics where available. "
        "Note the source (URL or publication name) for each finding."
    ),
    expected_output=(
        "A numbered list of 5-7 key findings. "
        "Each item: one sentence of fact + a brief source note in parentheses."
    ),
    agent=researcher,
)

print(f"Task defined for agent: {research_task.agent.role}")
print(f"Description (first 100 chars): {research_task.description[:100]}...")

In [ ]:
# ─── 4-B: Define the writing task ────────────────────────────────────────────
# context=[research_task] is the key line — it wires the two tasks together.

write_task = Task(
    description=(
        "Using the research findings provided in context, write a structured report. "
        "Target length: 200 words. "
        "Structure: Introduction (1 paragraph), Key Findings (3-5 bullet points), Conclusion (1 paragraph). "
        "Write for a developer audience: precise, no fluff."
    ),
    expected_output=(
        "A 200-word structured report with three clearly labelled sections: "
        "Introduction, Key Findings, Conclusion."
    ),
    agent=writer,
    context=[research_task],  # <-- passes researcher's output into this task
)

print(f"Task defined for agent: {write_task.agent.role}")
print(f"Context tasks: {[t.agent.role for t in write_task.context]}")

In [ ]:
# ─── 4-C: Inspect the full task chain ────────────────────────────────────────

tasks = [research_task, write_task]
print(f"{'#':>2}  {'Agent':25}  {'Context from':20}  Expected output (first 60 chars)")
print("─" * 100)
for i, t in enumerate(tasks, 1):
    ctx = ", ".join(c.agent.role for c in t.context) if t.context else "(none)"
    print(f"{i:>2}  {t.agent.role:25}  {ctx:20}  {t.expected_output[:60]}...")

---

## Part 5 — The Crew: Orchestrating Agents Through Tasks

---

A `Crew` is the container that runs agents through tasks. It owns two key concerns:

1. **Ordering**: which tasks run in what order
2. **Delegation**: which agent is responsible for which task

### Process Modes

| Mode | How it works | Use when |
|------|-------------|----------|
| `Process.sequential` | Tasks run one after another, left to right | Linear pipelines (default) |
| `Process.hierarchical` | A manager LLM decides task order and delegation | Complex, unpredictable workflows |

```python
from crewai import Process

# Sequential (default) — tasks run in the order they appear in `tasks`
Crew(agents=[...], tasks=[...], process=Process.sequential)

# Hierarchical — a manager agent plans and delegates automatically
Crew(agents=[...], tasks=[...], process=Process.hierarchical, manager_llm="gpt-4o")
```

For this workshop we use `sequential` — the researcher runs first, then the writer.

In [ ]:
# ─── 5-A: Assemble the Crew ───────────────────────────────────────────────────

from crewai import Crew, Process

crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, write_task],
    process=Process.sequential,
    verbose=True,
)

print("Crew assembled:")
print(f"  Agents:  {[a.role for a in crew.agents]}")
print(f"  Tasks:   {len(crew.tasks)} task(s) in sequential order")
print(f"  Process: sequential")

---

## Part 6 — Run the Crew

---

`crew.kickoff()` starts execution. With `verbose=True`, you see each agent's reasoning chain
as it runs — thoughts, tool calls, and intermediate outputs.

The return value is the **final task's output** — in this case, the writer's report.

> **Cost note:** this makes real LLM calls and at least one web search.
> Expect 3-5 API calls and roughly $0.01-$0.05 total depending on your model.

In [ ]:
# ─── 6-A: Kickoff the crew ────────────────────────────────────────────────────
# Watch the verbose output — it shows each agent's chain-of-thought in real time.

result = crew.kickoff()

print("\n" + "=" * 60)
print("FINAL REPORT")
print("=" * 60)
print(result)

In [ ]:
# ─── 6-B: Inspect usage metrics ──────────────────────────────────────────────
# CrewAI tracks token usage per run. Useful for cost estimation.

if hasattr(result, "token_usage"):
    usage = result.token_usage
    print("Token usage summary:")
    print(f"  Total tokens:       {usage.total_tokens:,}")
    print(f"  Prompt tokens:      {usage.prompt_tokens:,}")
    print(f"  Completion tokens:  {usage.completion_tokens:,}")
else:
    print("Token usage: not available in this CrewAI version.")
    print(f"Result type: {type(result)}")
    print(f"Result (first 200 chars): {str(result)[:200]}")

---

## Part 7 — Process Modes and the LangGraph Comparison

---

### Side-by-Side: Same Pipeline, Two Frameworks

```python
# ── LangGraph: explicit state machine ─────────────────────────────────────
from langgraph.graph import StateGraph, END
from typing import TypedDict

class ResearchState(TypedDict):
    topic: str
    findings: str
    report: str

def research_node(state):
    return {"findings": run_research(state["topic"])}

def write_node(state):
    return {"report": run_writer(state["findings"])}

workflow = StateGraph(ResearchState)
workflow.add_node("research", research_node)
workflow.add_node("write",    write_node)
workflow.set_entry_point("research")
workflow.add_edge("research", "write")
workflow.add_edge("write", END)
app = workflow.compile()
result = app.invoke({"topic": "AI agents"})


# ── CrewAI: declarative role assignment ───────────────────────────────────
from crewai import Agent, Task, Crew

researcher = Agent(role="Researcher", goal="...", backstory="...", tools=[web_search])
writer     = Agent(role="Writer",     goal="...", backstory="...")

research_task = Task(description="...", expected_output="...", agent=researcher)
write_task    = Task(description="...", expected_output="...", agent=writer, context=[research_task])

crew   = Crew(agents=[researcher, writer], tasks=[research_task, write_task])
result = crew.kickoff()
```

### When to Choose Each

| Scenario | Recommended |
|----------|-------------|
| Linear pipeline with role-based specialisation | CrewAI |
| Conditional branching (if X then Y else Z) | LangGraph |
| Human-in-the-loop interrupts | LangGraph |
| Parallel agent execution (crews within crews) | CrewAI |
| Custom retry and error recovery logic | LangGraph |
| Rapid prototyping of role-based systems | CrewAI |

---

## Exercise 1 — Run the Crew on Your Own Topic

Change `MY_TOPIC` and re-run the crew on a subject you are curious about.
Try:
- `"Retrieval-Augmented Generation in 2025"`
- `"Open-source LLMs vs GPT-4o in late 2025"`
- `"Autonomous AI agents in production: risks and benefits"`

**Goal:** Observe how the researcher adapts its searches based on the topic,
and notice how the writer's structure stays consistent across different subjects.

In [ ]:
# ── Exercise 1 Starter ────────────────────────────────────────────────────────
MY_TOPIC = "your topic here"  # TODO: change this

ex1_research = Task(
    description=f"Research '{MY_TOPIC}'. Gather 5-7 key facts with source references.",
    expected_output="A numbered list of key findings, each with a brief source note.",
    agent=researcher,
)
ex1_write = Task(
    description="Write a 200-word structured report: Introduction, Key Findings, Conclusion.",
    expected_output="A structured 200-word report with clearly labelled sections.",
    agent=writer,
    context=[ex1_research],
)
ex1_crew = Crew(
    agents=[researcher, writer],
    tasks=[ex1_research, ex1_write],
    verbose=False,  # set True to see step-by-step reasoning
)

if MY_TOPIC != "your topic here":
    print(ex1_crew.kickoff())
else:
    print("TODO: set MY_TOPIC to something you're curious about, then re-run this cell.")

---

## Exercise 2 — Add a Third Agent: the Editor

Extend the crew with an `Editor` agent who reviews the writer's draft and improves clarity.

**Steps:**
1. Define an `editor` `Agent` with `role="Editor"` and a suitable `goal` and `backstory`
2. Define an `edit_task` with `context=[write_task]` so it receives the writer's draft
3. Build a new `Crew` with three agents and three tasks in order

**Challenge:** What should the `expected_output` of the edit task be?
Think about what an editor actually delivers — a tracked-changes document? A final clean version?

In [ ]:
# ── Exercise 2 Starter ────────────────────────────────────────────────────────

editor = Agent(
    role="Editor",
    goal="TODO: what should the editor's goal be?",
    backstory="TODO: write a backstory that makes this agent a good editor.",
    verbose=True,
)

edit_task = Task(
    description="TODO: what should the editor do with the writer's draft?",
    expected_output="TODO: what does a good edit deliver?",
    agent=editor,
    context=[],  # TODO: which task's output should the editor receive?
)

# Uncomment when ready:
# three_crew = Crew(
#     agents=[researcher, writer, editor],
#     tasks=[research_task, write_task, edit_task],
#     verbose=True,
# )
# print(three_crew.kickoff())

---

## Part 8 ★ — Extensions: Custom Tools, Memory, Async (Bonus)

---

### Custom Tool Patterns

Any Python function becomes a tool with `@tool`. Common patterns:

```python
# Tool that calls an external API
@tool("GitHub Stars")
def get_github_stars(repo: str) -> str:
    """Get the star count for a GitHub repository (format: owner/repo)."""
    import requests
    r = requests.get(f"https://api.github.com/repos/{repo}")
    return str(r.json().get("stargazers_count", "unknown"))

# Tool that reads a local file
@tool("File Reader")
def read_file(path: str) -> str:
    """Read the contents of a local file. Use for project documentation."""
    from pathlib import Path
    return Path(path).read_text()
```

### Memory (Experimental)

Enable crew memory to persist facts across `kickoff()` calls:

```python
crew = Crew(
    agents=[researcher, writer],
    tasks=[research_task, write_task],
    memory=True,          # persist short-term + long-term memory
    embedder={"provider": "openai", "config": {"model": "text-embedding-3-small"}},
)
```

### Async Kickoff

For use inside async web servers (FastAPI, etc.):

```python
import asyncio

async def run_crew(topic: str):
    result = await crew.kickoff_async(inputs={"topic": topic})
    return result
```

### Hierarchical Process — Manager-Led Delegation

```python
from crewai import Process

# A manager LLM plans, delegates, and reviews — you don't wire tasks manually
crew = Crew(
    agents=[researcher, writer, editor],
    tasks=[broad_task],         # just one high-level task
    process=Process.hierarchical,
    manager_llm="gpt-4o",       # the planning / orchestration model
)
```

Use hierarchical when the number of sub-tasks is unknown ahead of time or when you want the
manager to decide whether to delegate or handle tasks directly.

In [ ]:
# ─── 8-A: Build a custom text-summariser tool ─────────────────────────────────
# This tool wraps a pure Python function — no external API needed.
# It demonstrates that tools can be computation, not just retrieval.

@tool("Text Summariser")
def summarise_text(text: str, max_words: int = 50) -> str:
    """Summarise a block of text to at most max_words words.
    Use when you have retrieved a long document and need a short digest.
    """
    words = text.split()
    if len(words) <= max_words:
        return text
    truncated = " ".join(words[:max_words])
    return truncated + "  [...truncated]"


# Test it without any LLM calls
long_text = "CrewAI is a framework for building multi-agent systems. " * 10
summary = summarise_text.run(long_text)
print(f"Original word count: {len(long_text.split())}")
print(f"Summary: {summary[:120]}")

---

## What's Next?

You now have a working CrewAI research crew and understand the three core primitives.
Here's where to go deeper:

### Try related examples in this repo
- **Example 12** — [Basic RAG Notebook](../12-basic-rag-notebook/rag_workbook.ipynb): add a retrieval tool so the researcher can search a private knowledge base, not just the public web
- **Example 16** — RAG Evaluation with RAGAS: measure the quality of retrieved context automatically
- **Example 14** — [SQL Agent](../14-sql-agent/README.md): give an agent the ability to write and execute SQL queries

### Extend what you built
- Add a **fact-checker** agent that verifies each of the researcher's claims before the writer proceeds
- Wire a `FileWriteTool` to the writer so the report is saved to disk automatically
- Switch to `Process.hierarchical` and observe how a manager LLM decomposes the same objective differently
- Enable `memory=True` and run the crew on the same topic twice — see if it recalls prior research

### Key takeaways

| Concept | What to remember |
|---------|------------------|
| `Agent` | Defines identity: role, goal, backstory, tools |
| `Task` | Defines work: description, expected output, assigned agent |
| `context=` | Passes one task's output into the next — the key to collaboration |
| `Crew` | Orchestrates agents through tasks; owns process mode |
| `@tool` | Wraps any Python function into a callable LLM tool |
| `Process.sequential` | Tasks run in order — simplest and most predictable |
| `Process.hierarchical` | Manager LLM plans and delegates — flexible, less predictable |

### Further reading
> Wu, Q., Bansal, G., et al. (2023). *AutoGen: Enabling Next-Gen LLM Applications via Multi-Agent Conversation.* https://arxiv.org/abs/2308.08155  
> Hong, S., Zheng, X., et al. (2023). *MetaGPT: Meta Programming for A Multi-Agent Collaborative Framework.* https://arxiv.org/abs/2308.00352  
> CrewAI docs — hierarchical crews, memory, async: https://docs.crewai.com  
> LangGraph vs CrewAI discussion: https://blog.langchain.dev

---

*Workshop complete. Open Example 12 for the RAG deep-dive, or Example 16 to put scores on your retrieval pipeline.*

---

## Exercise Answer Key

Use this section as a reference after attempting the exercises yourself.
These are sample solutions — not the only correct answers.

### Exercise 1 — Run the Crew on Your Own Topic

**What good output looks like:**
- The researcher should return a numbered list (1-7 items) with source notes in parentheses.
  If it returns prose instead of a list, tighten the `expected_output` field.
- The writer's report should have three clearly labelled sections: Introduction, Key Findings, Conclusion.
  If sections are missing, the `description` field needs to be more explicit.

**Debugging tip:** If the crew produces poor output, the most common cause is a vague `expected_output`.
The LLM treats `expected_output` as a quality bar — be specific about format, length, and structure.

In [ ]:
# Sample answer for Exercise 1 — well-specified tasks
SAMPLE_TOPIC = "Retrieval-Augmented Generation in 2025"

sample_research = Task(
    description=(
        f"Research '{SAMPLE_TOPIC}'. Search for recent developments, popular frameworks, "
        "benchmark results, and adoption numbers. Include source URLs or publication names."
    ),
    expected_output=(
        "A numbered list of exactly 5-7 findings. "
        "Format each as: 'N. [Fact]. (Source: [name or URL])'"
    ),
    agent=researcher,
)
sample_write = Task(
    description=(
        "Write a 200-word developer report using the research findings in context. "
        "Use three sections: **Introduction** (2 sentences), "
        "**Key Findings** (3-5 bullets), **Conclusion** (2 sentences)."
    ),
    expected_output=(
        "A Markdown-formatted report under 250 words with bold section headings: "
        "Introduction, Key Findings, Conclusion."
    ),
    agent=writer,
    context=[sample_research],
)

print("Sample tasks defined. Uncomment the lines below to run them.")
# sample_crew = Crew(agents=[researcher, writer], tasks=[sample_research, sample_write], verbose=False)
# print(sample_crew.kickoff())

### Exercise 2 — Add a Third Agent: the Editor

**Key insight on `context`:** The editor should receive `context=[write_task]` — it needs the
writer's draft, not the researcher's raw findings. Getting context chains right is the most
common source of bugs in multi-agent CrewAI systems.

**What good `expected_output` looks like for an editor:**
Be explicit — `"A revised 200-word report with the same three sections. Changes annotated in square brackets."`
is far better than `"An improved report"` because the LLM knows exactly what format to return.

In [ ]:
# Sample answer for Exercise 2 — three-agent crew

sample_editor = Agent(
    role="Editor",
    goal="Improve the clarity, accuracy, and conciseness of the writer's draft",
    backstory=(
        "You are a meticulous editor with 10 years of experience in technical publishing. "
        "You cut filler words, fix ambiguous phrasing, and ensure every claim is "
        "supported by the research. You never add new facts — only improve presentation."
    ),
    verbose=True,
)

sample_edit_task = Task(
    description=(
        "Review the writer's draft report provided in context. "
        "Fix: awkward phrasing, passive voice, redundant sentences, and any claims "
        "not supported by the research findings. Maintain the three-section structure."
    ),
    expected_output=(
        "A final polished report under 220 words. Same three sections: "
        "Introduction, Key Findings, Conclusion. No new facts added."
    ),
    agent=sample_editor,
    context=[write_task],  # <-- receives the writer's draft
)

print("Three-agent crew defined. Uncomment to run.")
# three_crew = Crew(
#     agents=[researcher, writer, sample_editor],
#     tasks=[research_task, write_task, sample_edit_task],
#     verbose=True,
# )
# print(three_crew.kickoff())